# kasb-crawler — EXAONE-3.5-7.8B QLoRA 파인튜닝 (Colab)

**목표**: 근거 없는 문단번호를 지어내 인용하는 문제(citation fabrication)를 줄이기 위해, RAFT 스타일로 만든 700개 학습 데이터(근거 있음 600 + 근거없음 거부 100)로 EXAONE-3.5-7.8B-Instruct를 QLoRA(4bit 양자화 상태에서 LoRA만 학습)로 파인튜닝합니다.

**이 노트북의 하이퍼파라미터는 전부 출처가 있습니다.** `[논문 확정값]`은 QLoRA/LoRA 원 논문에 실제로 적힌 숫자이고, `[실무 판단]`은 어느 논문에도 없어서 700개짜리 소규모 데이터셋 특성을 감안해 직접 고른 값입니다(과적합 방지 근거는 학습 중 loss 그래프와 마지막 30문항 평가로 실측 확인 — 논문이 보장해주지 않습니다).

| 하이퍼파라미터 | 이 노트북 값 | 근거 |
|---|---|---|
| 4bit 양자화 방식 | NF4 + Double Quantization, compute dtype bf16 | **[논문 확정값]** QLoRA 논문(Dettmers et al., 2023, arXiv:2305.14314) Section 3, Appendix B.2: "we use NF4 with double quantization and bf16 computation datatype" |
| LoRA rank r | 64 | **[논문 확정값]** 위 논문 Appendix B.2: "We set LoRA r = 64, α = 16" (7B~65B 전 모델 공통) |
| LoRA alpha | 16 | **[논문 확정값]** 위와 동일 문장 |
| LoRA dropout | 0.1 | **[논문 확정값]** 위 논문: 7B/13B급은 0.1, 33B/65B급은 0.05 — EXAONE 7.8B는 7B급이므로 0.1 |
| target_modules | 전체 linear layer (all-linear) | **[논문 확정값]** 위 논문 Section 4, Figure 2: "LoRA on all transformer layers is critical to match 16-bit [full fine-tuning] performance" — attention만 붙이면 성능이 못 미침을 ablation으로 확인 |
| 학습률(LR) | 2e-4, constant 스케줄 | **[논문 확정값]** 위 논문 Table 9 (7B 기준) + Appendix B.2: "we use a constant learning rate schedule" |
| 옵티마이저 | paged_adamw_32bit, adam_beta2=0.999 | **[논문 확정값]** 위 논문 Section 3(paged optimizer) + Appendix B.2(beta2=0.999) |
| max_grad_norm | 0.3 | **[논문 확정값]** 위 논문 Appendix B.2 |
| epoch 수 | 3 (매 epoch 체크포인트 저장) | **[실무 판단]** QLoRA 논문은 epoch이 아니라 step 수만 명시해 700개 규모에 대한 직접 근거가 없음. 참고용으로 LIMA 논문(Zhou et al., 2023)은 1,000개 데이터로 15 epoch을 돌리되 5~10 epoch 사이에서 사람이 직접 품질을 보고 체크포인트를 골랐음(perplexity가 생성 품질과 상관없었다고 명시). 저희는 held-out 30문항 평가가 이미 있으므로 3 epoch을 기본으로 하되 epoch마다 저장해서 실측으로 최적 체크포인트를 고릅니다 — "몇 epoch이 정답"이라는 주장은 어떤 논문에도 없으므로 실측으로 결정합니다. |
| batch size / gradient accumulation | 4 / 4 (실효 배치 16) | **[실무 판단]** 논문 값이 아니라 Colab GPU 메모리 제약에 맞춘 값. OOM 나면 batch를 줄이고 accumulation을 늘려 실효 배치는 유지하세요. |
| weight_decay, warmup | 0 (HF 기본값 그대로) | **[확인 안 됨]** QLoRA 논문 Table 9에 이 값들이 아예 없음. 임의로 숫자를 지어내지 않고 라이브러리 기본값을 그대로 둡니다. |

**중요 — 라이브러리 기본값 함정**: HuggingFace `BitsAndBytesConfig`의 기본값은 `bnb_4bit_quant_type="fp4"`(NF4 아님!), `bnb_4bit_use_double_quant=False`, `bnb_4bit_compute_dtype=torch.float32`입니다. 즉 **아무 설정 없이 4bit만 켜면 QLoRA 논문 설정이 재현되지 않습니다.** 아래 코드에서 전부 명시적으로 지정했으니 이 셀을 지우지 마세요.

## 0. GPU 확인
런타임 > 런타임 유형 변경에서 GPU(A100 권장, 없으면 L4/T4도 가능)를 먼저 켜세요.

In [1]:
!nvidia-smi

Tue Jul 14 15:51:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             51W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. 라이브러리 설치

In [2]:
!pip install -q -U "transformers>=5.0,<5.9" accelerate peft bitsandbytes trl datasets huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 98.7 MB/s eta 0:00:00


### 1-1. HuggingFace 다운로드 오류 예방 (Xet 비활성화)
큰 모델(수십 GB)을 받을 때 HuggingFace의 신규 다운로드 방식("Xet")에서 `403 Forbidden: SignatureError: invalid key pair id` 오류나 극도로 느린 다운로드가 실제로 보고되고 있습니다(HuggingFace 서버 쪽 인프라 이슈, huggingface/datasets GitHub 이슈 #8328 등). 환경변수 `HF_HUB_DISABLE_XET`만으로는 특정 버전에서 안 먹히는 버그도 보고돼 있어(huggingface_hub 이슈 #3266), 가장 확실한 방법은 `hf_xet` 패키지 자체를 지워 예전 방식(HTTP) 다운로드로 강제 전환하는 것입니다.

In [3]:
!pip uninstall -y hf_xet -q
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"  # 버전에 따라 무시될 수 있어 위 uninstall이 실질적인 안전장치

## 2. HuggingFace 로그인
`LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct`가 게이트(라이선스 동의) 모델일 수 있습니다. https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct 에서 먼저 라이선스에 동의한 뒤, 아래 셀에서 토큰(read 권한)으로 로그인하세요.

In [4]:
from huggingface_hub import notebook_login
notebook_login()

## 3. 학습 데이터 업로드
`raft_train_data_full.jsonl`(700건, kasb-crawler 세션 스크래치패드에서 생성한 파일)을 업로드하세요. 세션이 끊기면 사라지므로, 반복 실험할 계획이면 Google Drive에 올려두고 마운트하는 걸 권장합니다(아래 대안 셀 참고).

In [5]:
from google.colab import files
uploaded = files.upload()  # raft_train_data_full.jsonl 선택
DATA_PATH = "raft_train_data_full.jsonl"

Saving raft_train_data_full.jsonl to raft_train_data_full (2).jsonl


### (대안) Google Drive 마운트 — 체크포인트를 세션 종료 후에도 보존하고 싶다면 이 방식 권장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK_DIR = "/content/drive/MyDrive/kasb_lora"
os.makedirs(WORK_DIR, exist_ok=True)
# raft_train_data_full.jsonl을 미리 이 폴더(WORK_DIR)에 올려뒀다면:
# DATA_PATH = os.path.join(WORK_DIR, "raft_train_data_full.jsonl")

## 4. 데이터셋 로드 및 분할
700건 중 내부 검증(loss 모니터링)용으로 소량만 떼어냅니다. 최종 성능 판단은 이 검증셋이 아니라 **별도로 이미 제외해둔 held-out 30문항**(`exaone_baseline_results.jsonl`)으로만 합니다 — 이 30문항은 애초에 학습 데이터 생성 단계에서부터 빠져 있습니다.

In [6]:
import json, random
from datasets import Dataset

records = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]
print(f"전체 {len(records)}건")

rng = random.Random(42)
rng.shuffle(records)
N_VAL = 40  # 학습 중 loss 모니터링용 내부 검증 (최종 평가 아님)
val_records, train_records = records[:N_VAL], records[N_VAL:]
print(f"train={len(train_records)}  internal_val={len(val_records)}")

train_ds = Dataset.from_list(train_records)
val_ds = Dataset.from_list(val_records)

전체 700건
train=660  internal_val=40


## 5. 토크나이저 로드 + 채팅 템플릿으로 텍스트 포맷팅
EXAONE 토크나이저에 등록된 공식 chat template(`tokenizer.apply_chat_template`)을 그대로 써서, 모델이 원래 인스트럭션 튜닝될 때 본 형식과 최대한 똑같이 맞춥니다. 템플릿 문자열을 직접 만들어 쓰지 않는 이유: EXAONE의 정확한 특수 토큰(역할 구분자 등)을 추측으로 하드코딩하면 형식이 미묘하게 어긋날 위험이 있기 때문입니다.

In [7]:
from transformers import AutoTokenizer

BASE_MODEL = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_text(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return example

train_ds = train_ds.map(to_text, remove_columns=[c for c in train_ds.column_names if c != "text"])
val_ds = val_ds.map(to_text, remove_columns=[c for c in val_ds.column_names if c != "text"])

print("--- 렌더링된 예시 (형식 확인용) ---")
print(train_ds[0]["text"][:800])

Map:   0%|          | 0/660 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

--- 렌더링된 예시 (형식 확인용) ---
[|system|]너는 한국 회계기준 답변가다. 아래 '근거'만 사용해 한국어로 답한다. 근거에 없는 내용은 지어내지 말 것. 인용은 반드시 근거의 대괄호 식별자를 그대로 [식별자] 형태로 문장에 넣는다(예: [제1116호 문단 7]). 근거만으로 답할 수 없으면 다른 말 없이 정확히 '근거를 찾지 못했습니다.'만 출력한다.[|endofturn|]
[|user|]질문: 1 회사는 외주 건설공사로 유형자산을 취득하는 경우, 건설중인자산 취득원가에
건설자금에 대한 이자와 간접비 배부액을 포함하였다. 이 간접비에는 매출 발
생 부문의 직접비와 간접비를 제외한 모든 간접비(관리비를 포함한 포괄적 범
위의 인건비와 경비 등)가 포함되었다.
2 자가건설이 아닌 외주제작 방식으로 유형자산을 취득하는 경우, 도급금액 외에 발
생한 원가(간접비)를 해당 자산의 취득원가에 포함하는가?
2/5

근거:
[제1016호 문단 16] (kifrs_standards) 유형자산의 원가는 다음과 같이 구성된다.
⑴ 관세 및 환급불가능한 취득 관련 세금을 가산하고 매입할인과 리베이트를 차감한 구입가격
⑵ 경영진이 의도하는 방식으로 자산을 가동하는 데 필요한 장소와 상태에 이르게 하는 데 직접 관련되는 원가
⑶ 자산을 해체, 제거하거나 부지를 복구하는 데 소요될 것으로 최초에 추정되는 원가. 회사가 자산을 해체, 제거하거나 부지를 복구할 의무는 해당 유형자산을 취득한 시점에 또는 해당 유형자산을 특정기간 동안 재고자산 생산 이외의 목적으로 사용한 결과로서 발생한다.

[제1016호 문단 19] (kifrs_standards) 유형자산의 원가가


## 6. 4bit 양자화 설정 (QLoRA 논문 값 그대로)

In [8]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # 기본값 fp4가 아니라 반드시 nf4로 명시 (QLoRA 논문)
    bnb_4bit_use_double_quant=True,        # 기본값 False → True로 명시 (QLoRA 논문)
    bnb_4bit_compute_dtype=torch.bfloat16, # 기본값 float32 → bfloat16으로 명시 (QLoRA 논문)
)

In [10]:
import glob, re, inspect
from transformers.masking_utils import create_causal_mask

sig_params = list(inspect.signature(create_causal_mask).parameters.keys())
has_cache_position = "cache_position" in sig_params
print("현재 create_causal_mask 인자:", sig_params)

candidates = glob.glob("/root/.cache/huggingface/modules/transformers_modules/**/modeling_exaone.py", recursive=True)
print("찾은 파일:", candidates)
assert candidates, "캐시된 modeling_exaone.py를 못 찾음"

for path in candidates:
    src = open(path, encoding="utf-8").read()
    changed = False

    if "input_embeds=inputs_embeds" in src:
        src = src.replace("input_embeds=inputs_embeds", "inputs_embeds=inputs_embeds")
        changed = True
        print(f"[{path}] input_embeds -> inputs_embeds 치환")

    if not has_cache_position:
        pattern = re.compile(
            r"(attention_mask=attention_mask,\s*\n)(\s*)cache_position=cache_position,\s*\n(\s*)(past_key_values=past_key_values,)"
        )
        new_src, n = pattern.subn(r"\1\3\4", src)
        if n:
            src = new_src
            changed = True
            print(f"[{path}] cache_position 인자 {n}곳 제거(현재 버전 미지원)")

    if changed:
        open(path, "w", encoding="utf-8").write(src)
        print(f"저장 완료: {path}")
    else:
        print(f"[{path}] 변경 없음")


현재 create_causal_mask 인자: ['config', 'inputs_embeds', 'attention_mask', 'cache_position', 'past_key_values', 'position_ids', 'or_mask_function', 'and_mask_function']
찾은 파일: ['/root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_7_dot_8B_hyphen_Instruct/553ea250b9a5317231459279d5847d6cf955b9aa/modeling_exaone.py']
[/root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_7_dot_8B_hyphen_Instruct/553ea250b9a5317231459279d5847d6cf955b9aa/modeling_exaone.py] input_embeds -> inputs_embeds 치환
저장 완료: /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_7_dot_8B_hyphen_Instruct/553ea250b9a5317231459279d5847d6cf955b9aa/modeling_exaone.py


In [11]:
import time
from transformers import AutoModelForCausalLM

MAX_RETRIES = 30
model = None
for attempt in range(1, MAX_RETRIES + 1):
    try:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        print(f"성공! (시도 {attempt}회 만에)")
        break
    except OSError as e:
        print(f"[시도 {attempt}/{MAX_RETRIES}] 실패 — 30초 후 재시도 ({type(e).__name__})")
        time.sleep(30)

if model is None:
    raise RuntimeError("30회 재시도해도 실패 — HuggingFace 서버 문제가 아직 안 풀린 것 같습니다.")
model.config.use_cache = False


[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_7_dot_8B_hyphen_Instruct/553ea250b9a5317231459279d5847d6cf955b9aa/modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_7_dot_8B_hyphen_Instruct/553ea250b9a5317231459279d5847d6cf955b9aa/modeling_exaone.py.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


성공! (시도 1회 만에)


## 7. 베이스 모델 로드 (4bit)

In [14]:
import time

MAX_RETRIES = 30
model = None
for attempt in range(1, MAX_RETRIES + 1):
    try:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        print(f"성공! (시도 {attempt}회 만에)")
        break
    except OSError as e:
        print(f"[시도 {attempt}/{MAX_RETRIES}] 실패 — 30초 후 재시도 ({type(e).__name__})")
        time.sleep(30)

if model is None:
    raise RuntimeError("30회 재시도해도 실패 — HuggingFace 서버 문제가 아직 안 풀린 것 같습니다.")
model.config.use_cache = False


NameError: name 'AutoModelForCausalLM' is not defined

## 8. LoRA 설정 (QLoRA 논문 값 그대로) + 학습 준비

In [12]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch.nn as nn, types

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# EXAONE 커스텀 코드가 get_input_embeddings를 구현 안 해서 생기는 문제 우회:
# 실제 임베딩 레이어(nn.Embedding)를 직접 찾아 get/set_input_embeddings를 붙여준다.
def _find_embedding_module(m):
    for name, module in m.named_modules():
        if isinstance(module, nn.Embedding):
            return name, module
    return None, None

_emb_name, _emb_module = _find_embedding_module(model)
print("찾은 임베딩 레이어:", _emb_name)
assert _emb_name is not None, "임베딩 레이어를 못 찾았습니다 — model 구조를 print(model)로 확인해주세요"

def _get_input_embeddings(self):
    return _emb_module
def _set_input_embeddings(self, value):
    parent = self
    *path, last = _emb_name.split(".")
    for p in path:
        parent = getattr(parent, p)
    setattr(parent, last, value)

model.get_input_embeddings = types.MethodType(_get_input_embeddings, model)
model.set_input_embeddings = types.MethodType(_set_input_embeddings, model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


[transformers] ExaoneForCausalLM does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


찾은 임베딩 레이어: transformer.wte
trainable params: 167,772,160 || all params: 7,986,221,056 || trainable%: 2.1008


`trainable params` 비율이 몇 %인지 여기서 꼭 확인하세요(QLoRA 7B 기준으로 대략 전체 파라미터의 1~3% 수준이 나오면 정상입니다 — r=64/all-linear 조합에서 크게 벗어나면 target_modules가 EXAONE 아키텍처에서 의도대로 안 잡혔을 수 있으니 `model.print_trainable_parameters()` 출력과 함께 확인 부탁드립니다).

## 9. 학습 설정 (SFTConfig)

In [13]:
from trl import SFTConfig

OUTPUT_DIR = "/content/drive/MyDrive/kasb_lora/checkpoints" if 'WORK_DIR' in dir() else "./checkpoints"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                  # [실무 판단] 위 근거표 참고 — epoch별 저장 후 실측으로 최적 epoch 선택
    per_device_train_batch_size=4,       # [실무 판단] Colab GPU 메모리에 맞춘 값, OOM이면 줄이세요
    gradient_accumulation_steps=4,       # 실효 배치 = 4*4 = 16
    learning_rate=2e-4,                  # [논문 확정값] QLoRA Table 9 (7B)
    lr_scheduler_type="constant",        # [논문 확정값] QLoRA Appendix B.2
    optim="paged_adamw_32bit",           # [논문 확정값] QLoRA의 paged optimizer
    adam_beta2=0.999,                    # [논문 확정값] QLoRA Appendix B.2
    max_grad_norm=0.3,                   # [논문 확정값] QLoRA Appendix B.2
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,                  # epoch 3개 체크포인트 전부 보존 (사후 비교용)
    max_length=4096,                     # 데이터 중 가장 긴 예시(약 14000자, distractor 포함)를 안전하게 담기 위한 여유값
    packing=False,                       # 예시별로 정답 경계가 뚜렷해야 하므로 packing 미사용
    dataset_text_field="text",
    report_to="none",
)

## 10. 학습 실행

In [14]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/660 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/660 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/660 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/660 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.130069,1.098926,1.108608,1089531.000000,0.730052
2,0.939994,0.984211,0.949303,2179062.000000,0.752280
3,0.782416,0.900087,0.895400,3268593.000000,0.771811


TrainOutput(global_step=126, training_loss=1.0322769218020968, metrics={'train_runtime': 2594.8873, 'train_samples_per_second': 0.763, 'train_steps_per_second': 0.049, 'total_flos': 2.0963557428579533e+17, 'train_loss': 1.0322769218020968, 'epoch': 3.0})

학습 로그의 `loss`(train)와 `eval_loss`(내부 검증)를 epoch마다 비교하세요. train_loss는 계속 떨어지는데 eval_loss가 다시 올라가기 시작하면 그 epoch부터 과적합입니다 — 그 이전 epoch 체크포인트를 최종 후보로 우선 고려하세요(단, 최종 판단은 아래 held-out 30문항 평가로 합니다. LIMA 논문도 "perplexity가 실제 답변 품질과 상관 없었다"고 명시했으니 loss만으로 확정하지 마세요).

## 11. LoRA 어댑터 저장

In [15]:
ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("저장 완료:", ADAPTER_DIR)

저장 완료: ./checkpoints/final_adapter


## 12. 병합(merge) — LoRA 어댑터를 원본 가중치에 합치기

학습은 4bit로 했지만, 배포용 가중치를 만들려면 **양자화 안 된 상태(bf16)로 베이스 모델을 다시 로드**해서 그 위에 LoRA를 합쳐야 합니다(4bit 가중치에는 직접 병합이 안 됩니다). 이 단계는 GPU 메모리를 더 많이 씁니다(7.8B bf16 ≈ 15.6GB) — A100(40GB) 권장, T4(16GB)는 부족할 수 있습니다.

In [86]:
!pip uninstall -y torchao -q
!pip uninstall -y hf_xet -q
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import gc, torch
for _v in ["model", "trainer"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch.nn as nn, types

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)

def _find_embedding_module(m):
    for name, module in m.named_modules():
        if isinstance(module, nn.Embedding):
            return name, module
    return None, None

_emb_name, _emb_module = _find_embedding_module(base)
print("찾은 임베딩 레이어:", _emb_name)
if _emb_name is not None:
    def _get_input_embeddings(self):
        return _emb_module
    def _set_input_embeddings(self, value):
        parent = self
        *path, last = _emb_name.split(".")
        for p in path:
            parent = getattr(parent, p)
        setattr(parent, last, value)
    base.get_input_embeddings = types.MethodType(_get_input_embeddings, base)
    base.set_input_embeddings = types.MethodType(_set_input_embeddings, base)

merged = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = merged.merge_and_unload()

MERGED_DIR = f"{OUTPUT_DIR}/merged_fp16"
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("병합 완료:", MERGED_DIR)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

OSError: There was a specific connection error when trying to load LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct:
(Request ID: 01KXGW7S04328XE4RXNY1QHDFE)

403 Forbidden: None.
Cannot access content at: https://us.gcp.cdn.hf.co/xet-bridge-us/674c617599dbb7a9a7f76420/c5a4683ecbcc15b8663a11d49a9a6b83ffb5c8cd6c6402f58aaa5ec311dcad8c?user_id=6997cb5b371eb51e83014d13&X-Xet-Cas-Uid=6997cb5b371eb51e83014d13&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model-00002-of-00007.safetensors%3B+filename%3D%22model-00002-of-00007.safetensors%22%3B&Expires=1784055255&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjc0YzYxNzU5OWRiYjdhOWE3Zjc2NDIwL2M1YTQ2ODNlY2JjYzE1Yjg2NjNhMTFkNDlhOWE2YjgzZmZiNWM4Y2Q2YzY0MDJmNThhYWE1ZWMzMTFkY2FkOGNcXD91c2VyX2lkPTY5OTdjYjViMzcxZWI1MWU4MzAxNGQxMyZYLVhldC1DYXMtVWlkPTY5OTdjYjViMzcxZWI1MWU4MzAxNGQxMyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4NDA1NTI1NX19fV19&Sig

In [20]:
import gc, torch
for _v in ["model", "trainer"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()


## 13. GGUF 변환 + 양자화 (Ollama 배포용)

`mlx_lm`의 GGUF 변환은 EXAONE 아키텍처를 지원하지 않는 것으로 이전에 확인했으므로, `llama.cpp`의 `convert_hf_to_gguf.py`를 씁니다.

In [47]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

fatal: destination path 'llama.cpp' already exists and is not an empty directory.


In [48]:
!python llama.cpp/convert_hf_to_gguf.py "{MERGED_DIR}" --outfile exaone_lora_merged_f16.gguf --outtype f16

INFO:hf-to-gguf:Loading model: merged_fp16
 You can inspect the repository content at https://hf.co/checkpoints/merged_fp16.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.
INFO:hf-to-gguf:Model architecture: ExaoneForCausalLM
 You can inspect the repository content at https://hf.co/checkpoints/merged_fp16.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> F16, shape = {4096, 102400}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> F16, shape = {4096, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,    torch.bfloat16 --> F16, shape = {4096, 4096}
INFO:hf-to-gguf:blk.0.attn_q.weight,         torch.bfloa

In [72]:
import os, torch, gc, types, shutil, glob

# 1. HuggingFace Xet 강제 비활성화 및 캐시 정리
!pip uninstall -y hf_xet -q
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# 기존에 생성되다 만 불완전한 파일 정리
if os.path.exists("exaone_lora_merged_f16.gguf"): os.remove("exaone_lora_merged_f16.gguf")
shutil.rmtree(MERGED_DIR, ignore_errors=True)

# 2. 병합된 모델 재생성
print("병합 모델 생성 중...")
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch.nn as nn

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# EXAONE 임베딩 처리
_emb_name, _emb_module = next((n, m) for n, m in base.named_modules() if isinstance(m, nn.Embedding))
base.get_input_embeddings = types.MethodType(lambda self: _emb_module, base)

merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

del base, merged; gc.collect(); torch.cuda.empty_cache()

# 3. GGUF 변환 (15.6GB 파일 생성)
print("GGUF 변환 시작 (F16)...")
os.environ['TMPDIR'] = '/content'
!python llama.cpp/convert_hf_to_gguf.py "{MERGED_DIR}" --outfile exaone_lora_merged_f16.gguf --outtype f16

# 4. 변환 완료 후 용량 확인 및 디스크 정리
if os.path.exists("exaone_lora_merged_f16.gguf"):
    file_size = os.path.getsize("exaone_lora_merged_f16.gguf") / (1024**3)
    print(f"변환 완료: {file_size:.2f} GB")
    if file_size > 14:
        print("원본 모델 폴더를 삭제하여 공간을 확보합니다.")
        shutil.rmtree(MERGED_DIR, ignore_errors=True)
        !df -h /content
    else:
        print("경고: 파일 용량이 부족합니다. 다시 시도해 주세요.")

병합 모델 생성 중...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

ValueError: Can't find 'adapter_config.json' at './checkpoints/final_adapter'

In [77]:
import os, torch, gc, types, shutil, glob
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch.nn as nn

# 1. 강력한 디스크 정리 (찌꺼기 파일 제거)
print("디스크 찌꺼기 파일 정리 중...")
# (1) HuggingFace 임시 다운로드 폴더(staging) 삭제
!rm -rf /root/.cache/huggingface/hub/tmp*
# (2) 미완성 GGUF 파일 및 이전 시도 파일 삭제
for f in glob.glob("*.gguf"):
    print(f"삭제 중: {f}")
    os.remove(f)
# (3) 기존 병합 폴더 정리
shutil.rmtree("./checkpoints/merged_full", ignore_errors=True)
shutil.rmtree("./checkpoints/merged_fp16", ignore_errors=True)
# (4) 시스템 임시 파일 정리
!rm -rf /tmp/*
!apt-get clean

print("정리 후 디스크 상태:")
!df -h /content

# 2. 구글 드라이브 마운트 및 경로 설정
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

ADAPTER_PATH = "/content/drive/MyDrive/adaptor"
MERGED_PATH = "./checkpoints/merged_full"

if not os.path.exists(ADAPTER_PATH):
    print(f"❌ 오류: {ADAPTER_PATH} 경로를 찾을 수 없습니다.")
else:
    # 3. 모델 병합 (BF16)
    print("모델 병합 시작...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
    )

    # EXAONE 임베딩 메서드 주입
    _find_emb = lambda m: next(((n, module) for n, module in m.named_modules() if isinstance(module, nn.Embedding)), (None, None))
    _emb_name, _emb_module = _find_emb(base)
    if _emb_name: base.get_input_embeddings = types.MethodType(lambda self: _emb_module, base)

    merged = PeftModel.from_pretrained(base, ADAPTER_PATH).merge_and_unload()
    merged.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)
    del base, merged; gc.collect(); torch.cuda.empty_cache()

    # 4. GGUF 변환
    print("GGUF 변환 중 (15.6GB 생성 예정)...")
    !python llama.cpp/convert_hf_to_gguf.py "{MERGED_PATH}" --outfile exaone_raw_f16.gguf --outtype f16

    # 5. 메타데이터 수정 및 양자화
    import sys
    sys.path.insert(0, "llama.cpp/gguf-py")
    import gguf

    if os.path.exists("exaone_raw_f16.gguf") and os.path.getsize("exaone_raw_f16.gguf") > 14 * (1024**3):
        reader = gguf.GGUFReader("exaone_raw_f16.gguf", "r")
        writer = gguf.GGUFWriter("exaone_fixed_f16.gguf", arch="exaone")
        for field in reader.fields.values():
            writer.add_key_value(field.name, field.parts[-1], field.types[-1])
        writer.add_float32("exaone.attention.layer_norm_rms_epsilon", 1e-05)
        for tensor in reader.tensors:
            writer.add_tensor(tensor.name, tensor.data, raw_shape=tensor.shape, raw_dtype=tensor.tensor_type)
        writer.write_config_file("exaone_fixed_f16.gguf")
        writer.close()

        print("최종 Q4_K_M 양자화 진행...")
        !./llama.cpp/build/bin/llama-quantize exaone_fixed_f16.gguf exaone_lora_q4_k_m.gguf Q4_K_M

        if os.path.exists("exaone_lora_q4_k_m.gguf"):
            print("🎉 완료! exaone_lora_q4_k_m.gguf 생성 성공.")
            os.remove("exaone_raw_f16.gguf")
            os.remove("exaone_fixed_f16.gguf")
            shutil.rmtree(MERGED_PATH, ignore_errors=True)
    else:
        print("❌ 여전히 용량이 부족하여 변환이 중단되었습니다.")

디스크 찌꺼기 파일 정리 중...
삭제 중: exaone_raw_f16.gguf
정리 후 디스크 상태:
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   99G   15G  88% /
모델 병합 시작...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/adaptor/adapter_config.json'

In [78]:
from google.colab import drive
import os

# 드라기브 러느타기마거느 해개르르류ᄅ 해개르르류ᄅ 가시 마거느트
print("드라기브 레마거느트 시도 버나...")
drive.mount('/content/drive', force_remount=True)

ADAPTER_PATH = "/content/drive/MyDrive/adaptor"
if os.path.exists(ADAPTER_PATH):
    print(f"✅ 드라기브 겨녀ᄅ 보개 서개: {ADAPTER_PATH}")
else:
    print("❀ 녀류: 드라기브 마거느트 해 겨녀ᄅ르ᄅ 모스하거나 거다프터 포래더가 너버스ᄆ니다.")

드라기브 레마거느트 시도 버나...
Mounted at /content/drive
✅ 드라기브 겨녀ᄅ 보개 서개: /content/drive/MyDrive/adaptor


In [84]:
import os, torch, gc, types, shutil, glob, sys
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch.nn as nn

# 1. 환경 설정 및 Xet 비활성화
!pip uninstall -y hf_xet -q
os.environ["HF_HUB_DISABLE_XET"] = "1"

ADAPTER_PATH = "/content/drive/MyDrive/adaptor"
MERGED_PATH = "./merged_temp"
RAW_GGUF = "exaone_raw_f16.gguf"
FIXED_GGUF = "exaone_fixed_f16.gguf"
FINAL_GGUF = "exaone_lora_q4_k_m.gguf"

def run_final_pipeline():
    # 디스크 공간 최대로 확보
    print("Cleaning disk space to maximum...")
    !rm -rf /root/.cache/huggingface/hub/*
    !rm -rf /root/.cache/huggingface/modules/*
    !rm -rf ./checkpoints/*
    !rm -rf ./merged_temp
    for f in glob.glob("*.gguf"):
        if os.path.exists(f): os.remove(f)

    # 2. 모델 병합
    print("Step 1: Merging (CPU Mode for Stability)...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True
    )

    _find_emb = lambda m: next(((n, module) for n, module in m.named_modules() if isinstance(module, nn.Embedding)), (None, None))
    _emb_name, _emb_module = _find_emb(base)
    if _emb_name: base.get_input_embeddings = types.MethodType(lambda self: _emb_module, base)

    model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    merged = model.merge_and_unload()

    print("Saving merged model...")
    merged.save_pretrained(MERGED_PATH, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_PATH)

    del base, model, merged; gc.collect(); torch.cuda.empty_cache()

    # 3. GGUF 변환
    print("Step 2: Converting to F16 GGUF...")
    !python llama.cpp/convert_hf_to_gguf.py "{MERGED_PATH}" --outfile {RAW_GGUF} --outtype f16

    if os.path.exists(RAW_GGUF) and os.path.getsize(RAW_GGUF) > 10*(1024**3):
        shutil.rmtree(MERGED_PATH, ignore_errors=True)
        !rm -rf /root/.cache/huggingface/hub/*

        # 4. 메타데이터 수정
        print("Step 3: Fixing EXAONE Metadata...")
        sys.path.insert(0, "llama.cpp/gguf-py")
        import gguf
        reader = gguf.GGUFReader(RAW_GGUF, "r")
        writer = gguf.GGUFWriter(FIXED_GGUF, arch="exaone")
        for field in reader.fields.values():
            writer.add_key_value(field.name, field.parts[-1], field.types[-1])
        writer.add_float32("exaone.attention.layer_norm_rms_epsilon", 1e-05)
        for tensor in reader.tensors:
            writer.add_tensor(tensor.name, tensor.data, raw_shape=tensor.shape, raw_dtype=tensor.tensor_type)
        writer.write_config_file(FIXED_GGUF)
        writer.close()
        os.remove(RAW_GGUF)

        # 5. 양자화
        print("Step 4: Quantizing to Q4_K_M...")
        !./llama.cpp/build/bin/llama-quantize {FIXED_GGUF} {FINAL_GGUF} Q4_K_M

        if os.path.exists(FINAL_GGUF):
            print(f"🎉 SUCCESS! Final model: {FINAL_GGUF}")
            os.remove(FIXED_GGUF)
            !df -h /content
        else:
            print("❌ Quantization failed.")
    else:
        print("❌ Conversion failed or incomplete file created.")

run_final_pipeline()

Cleaning disk space to maximum...
Step 1: Merging (CPU Mode for Stability)...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

configuration_exaone.py:   0%|          | 0.00/11.3k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/4.96M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

OSError: There was a specific connection error when trying to load LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct:
(Request ID: 01KXGVN9768YHJXA404MFK66YM)

403 Forbidden: None.
Cannot access content at: https://us.gcp.cdn.hf.co/xet-bridge-us/674c617599dbb7a9a7f76420/c5a4683ecbcc15b8663a11d49a9a6b83ffb5c8cd6c6402f58aaa5ec311dcad8c?user_id=6997cb5b371eb51e83014d13&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model-00002-of-00007.safetensors%3B+filename%3D%22model-00002-of-00007.safetensors%22%3B&X-Xet-Cas-Uid=6997cb5b371eb51e83014d13&Expires=1784054649&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjc0YzYxNzU5OWRiYjdhOWE3Zjc2NDIwL2M1YTQ2ODNlY2JjYzE1Yjg2NjNhMTFkNDlhOWE2YjgzZmZiNWM4Y2Q2YzY0MDJmNThhYWE1ZWMzMTFkY2FkOGNcXD91c2VyX2lkPTY5OTdjYjViMzcxZWI1MWU4MzAxNGQxMyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD02OTk3Y2I1YjM3MWViNTFlODMwMTRkMTMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4NDA1NDY0OX19fV19&Sig

In [57]:
# 파일 삭제를 뒤로 미룹니다.
print("파일 삭제를 건너뛰고 메타데이터 수정을 먼저 진행합니다.")
!df -h /content

파일 삭제를 건너뛰고 메타데이터 수정을 먼저 진행합니다.
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  113G     0 100% /


In [49]:
# llama-quantize 빌드 (CPU 빌드로 충분 — 양자화 자체는 GPU 불필요)
!cmake -B llama.cpp/build -S llama.cpp -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --target llama-quantize -j 4

CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.16.0
-- ggml commit:  bf2c86d
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.2s)
-- Generating done (0.4s)
-- Build files have been written to: /content/llama.cpp/build
[  2%] Built target llama-common-base
[  2%] Built target cpp-httplib
[  6%] Built target ggml-base
[ 13%] Built target ggml-cpu
[ 13%] Built target ggml
[ 86%] Built target llama
[100%] Built target llama-common
[100%] Built target llama-quantize-impl
[100%] Built target llama-quantize


In [50]:
!./llama.cpp/build/bin/llama-quantize exaone_lora_merged_f16.gguf exaone_lora_q4_k_m.gguf Q4_K_M

llama_print_build_info: build = 1 (bf2c86d)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing 'exaone_lora_merged_f16.gguf' to 'exaone_lora_q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 27 key-value pairs and 292 tensors from exaone_lora_merged_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = exaone
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 7.8B
llama_model_loader: - kv   4:                         exaone.block_count u32              = 32
llama_model_loader: - kv   5:                      exaone.context_len

In [51]:
import os
print("현재 폴더:", os.getcwd())
print("변환된 f16 gguf 있음?:", os.path.exists("exaone_lora_merged_f16.gguf"))
print("llama-quantize 빌드된 실행파일 있음?:", os.path.exists("llama.cpp/build/bin/llama-quantize"))
print("최종 q4 gguf 있음?:", os.path.exists("exaone_lora_q4_k_m.gguf"))
print("현재 폴더 내 gguf 관련 파일들:", [f for f in os.listdir(".") if "gguf" in f.lower()])


현재 폴더: /content
변환된 f16 gguf 있음?: True
llama-quantize 빌드된 실행파일 있음?: True
최종 q4 gguf 있음?: False
현재 폴더 내 gguf 관련 파일들: ['exaone_lora_merged_f16.gguf']


In [67]:
import os, shutil, glob

# 1. 15.6GB GGUF 파일이 있는지 먼저 확인 (이게 있어야 다음 단계 진행 가능)
target_gguf = "exaone_lora_merged_f16.gguf"
if os.path.exists(target_gguf):
    size_gb = os.path.getsize(target_gguf) / (1024**3)
    print(f"확인: {target_gguf} ({size_gb:.2f} GB) 존재함.")

# 2. 공간을 점유하는 원본 모델 폴더 및 체크포인트 강제 삭제
print("대용량 폴더 삭제 중...")
shutil.rmtree("./checkpoints/merged_fp16", ignore_errors=True)
shutil.rmtree("./checkpoints/final_adapter", ignore_errors=True)
for path in glob.glob("./checkpoints/checkpoint-*"):
    shutil.rmtree(path, ignore_errors=True)

# 3. 파이프라인 캐시 및 시스템 임시파일 정리
!rm -rf /root/.cache/huggingface/hub/*
!apt-get clean

print("삭제 후 디스크 상태:")
!df -h /content

확인: exaone_lora_merged_f16.gguf (4.63 GB) 존재함.
대용량 폴더 삭제 중...
삭제 후 디스크 상태:
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   69G   45G  61% /


In [68]:
import sys, os
sys.path.insert(0, "llama.cpp/gguf-py")

# gguf_new_metadata.py의 경로를 정확히 찾아 로드
import gguf

SRC_GGUF = "exaone_lora_merged_f16.gguf"
FIXED_GGUF = "exaone_lora_merged_f16_fixed.gguf"

# 메타데이터 수정을 위한 헬퍼 함수 (RMSNorm Epsilon 주입)
reader = gguf.GGUFReader(SRC_GGUF, "r")
arch = reader.get_field(gguf.Keys.General.ARCHITECTURE).contents()
existing_eps = reader.get_field(f"{arch}.attention.layer_norm_epsilon").contents()

print(f"Fixing Metadata: {arch}, Epsilon: {existing_eps}")

# 신규 writer 생성하여 복사
writer = gguf.GGUFWriter(FIXED_GGUF, arch=arch)

# 모든 텐서와 필드 복사하며 누락된 RMS 필드 추가
for field in reader.fields.values():
    writer.add_key_value(field.name, field.parts[-1], field.types[-1])

# 필수 필드 주입
writer.add_float32(f"{arch}.attention.layer_norm_rms_epsilon", existing_eps)

# 텐서 복사
for tensor in reader.tensors:
    writer.add_tensor(tensor.name, tensor.data, raw_shape=tensor.shape, raw_dtype=tensor.tensor_type)

writer.write_config_file(FIXED_GGUF)
writer.close()

print(f"완료: {FIXED_GGUF} 생성됨. 이제 양자화를 실행하세요.")

ValueError: cannot reshape array of size 0 into shape (14336,4096)

In [40]:
import subprocess
result = subprocess.run(
    ["./llama.cpp/build/bin/llama-quantize", "exaone_lora_merged_f16.gguf", "exaone_lora_q4_k_m.gguf", "Q4_K_M"],
    capture_output=True, text=True
)
print("종료 코드:", result.returncode)
print("--- stdout ---")
print(result.stdout[-3000:])
print("--- stderr ---")
print(result.stderr[-3000:])


종료 코드: 1
--- stdout ---

--- stderr ---
                general.architecture str              = exaone
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 7.8B
llama_model_loader: - kv   4:                         exaone.block_count u32              = 32
llama_model_loader: - kv   5:                      exaone.context_length u32              = 32768
llama_model_loader: - kv   6:                    exaone.embedding_length u32              = 4096
llama_model_loader: - kv   7:                 exaone.feed_forward_length u32              = 14336
llama_model_loader: - kv   8:                exaone.attention.head_count u32              = 32
llama_model_loader: - kv   9:             exaone.attention.head_count_kv u32              = 8
llama_model_loader: -

In [58]:
import sys, os
sys.path.insert(0, "llama.cpp/gguf-py")
sys.path.insert(0, "llama.cpp/gguf-py/gguf/scripts")
import gguf
# gguf_new_metadata가 경로에 없는 경우를 대비해 직접 구현하거나 확인이 필요합니다.
from gguf_new_metadata import copy_with_new_metadata, MetadataDetails

SRC_GGUF = "exaone_lora_merged_f16.gguf"
FIXED_GGUF = "exaone_lora_merged_f16_fixed.gguf"

if not os.path.exists(SRC_GGUF):
    print(f"오류: {SRC_GGUF} 파일이 없습니다. 이전 단계(convert_hf_to_gguf)를 다시 실행하여 파일을 생성해야 합니다.")
else:
    reader = gguf.GGUFReader(SRC_GGUF, "r")
    arch = reader.get_field(gguf.Keys.General.ARCHITECTURE).contents()
    existing_eps = reader.get_field(f"{arch}.attention.layer_norm_epsilon").contents()
    print("기존 layer_norm_epsilon 값:", existing_eps, "/ architecture:", arch)

    new_metadata = {
        f"{arch}.attention.layer_norm_rms_epsilon": MetadataDetails(gguf.GGUFValueType.FLOAT32, existing_eps),
    }

    writer = gguf.GGUFWriter(FIXED_GGUF, arch=arch, endianess=reader.endianess)
    alignment_field = reader.get_field(gguf.Keys.General.ALIGNMENT)
    if alignment_field is not None:
        writer.data_alignment = alignment_field.contents()

    copy_with_new_metadata(reader, writer, new_metadata, remove_metadata=[])
    print("메타데이터 보강 완료:", FIXED_GGUF)

오류: exaone_lora_merged_f16.gguf 파일이 없습니다. 이전 단계(convert_hf_to_gguf)를 다시 실행하여 파일을 생성해야 합니다.


In [52]:
import shutil
shutil.copy("exaone_lora_q4_k_m.gguf", f"{OUTPUT_DIR}/exaone_lora_q4_k_m.gguf")
print("Drive에 저장 완료 — 로컬 맥에서 Drive 동기화 후 다운로드하세요 (직접 다운로드는 4~5GB라 느릴 수 있음)")

FileNotFoundError: [Errno 2] No such file or directory: 'exaone_lora_q4_k_m.gguf'

## 14. (로컬 맥에서 실행) Ollama에 새 태그로 배포

**아래는 Colab이 아니라 로컬 맥 터미널에서 실행하는 명령입니다.** 기존 `exaone3.5:7.8b` 태그는 롤백용으로 그대로 남겨두고, 새 태그로 추가합니다.

```bash
# 1) 기존 태그의 template/parameter를 그대로 재사용하기 위해 먼저 확인
ollama show exaone3.5:7.8b --modelfile > /tmp/exaone_base.modelfile
cat /tmp/exaone_base.modelfile

# 2) 새 Modelfile 작성 — FROM 줄만 새 gguf로 바꾸고 TEMPLATE/PARAMETER/SYSTEM은 위에서 확인한 걸 그대로 복사
cat > /tmp/exaone_lora.modelfile <<'EOF'
FROM /path/to/exaone_lora_q4_k_m.gguf
# (여기에 exaone_base.modelfile의 TEMPLATE/PARAMETER 블록을 그대로 붙여넣기)
EOF

# 3) 새 태그로 생성 (원본 exaone3.5:7.8b는 그대로 보존됨)
ollama create exaone3.5-lora:v1 -f /tmp/exaone_lora.modelfile
```

## 15. 필수 — held-out 30문항 재평가

**이 단계 없이는 LoRA가 실제로 도움이 됐는지 알 수 없습니다.** `KASB_LOCAL_MODEL=exaone3.5-lora:v1` 환경변수로 `exaone_baseline_eval.py`와 같은 패턴의 스크립트를 다시 돌려서, 기존 베이스라인(거부율 33%/10건, 평균 Faithfulness 0.900)과 비교하세요.

- Faithfulness가 베이스라인보다 오르고 거부율이 과도하게(예: 60%+) 튀지 않으면 → 성공, 새 태그를 실제로 스왑
- Faithfulness는 안 올랐는데 거부율만 급등했으면 → v2 프롬프트 실험 때와 같은 '인용 강박' 재발 가능성, 해당 epoch은 폐기하고 다른 epoch 체크포인트로 재시도
- 개선이 없으면 → 700개 규모 데이터로는 부족했거나 하이퍼파라미터 재탐색 필요 (솔직하게 실패로 기록)

```bash
KASB_LOCAL_MODEL=exaone3.5-lora:v1 python exaone_after_lora_eval.py
```